In [1]:
# CPU версия парсера 
import os
import sys
import glob
import time
import chromadb
from concurrent.futures import ProcessPoolExecutor
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from hypothesis_factory.ingestion import PyMuPDFReader

load_dotenv()
API_KEY = os.getenv("YANDEX_API_KEY")
FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
if not API_KEY or not FOLDER_ID:
    raise ValueError("Ключи не найдены! Проверь файл .env")


def parse_one(pdf_path: str):
    """Функция для отдельного процесса: парсит один PDF и возвращает чанки + время"""
    reader = PyMuPDFReader(extract_tables=False)
    t0 = time.time()
    chunks = reader.read(pdf_path)
    dt = time.time() - t0
    return pdf_path, chunks, dt


if __name__ == "__main__":
    print("Загрузка модели эмбеддингов (GPU)...")
    embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda')

    print("Инициализация ChromaDB...")
    client = chromadb.Client()
    try:
        client.delete_collection("materials_db_notebook")
    except Exception:
        pass
    collection = client.create_collection("materials_db_notebook")

    raw_data_dir = os.path.join(project_root, "data", "raw")
    pdf_files = glob.glob(os.path.join(raw_data_dir, "*.pdf"))
    if not pdf_files:
        raise ValueError(f"PDF файлы не найдены в папке {raw_data_dir}!")

    print(f"\nНачинаем парсинг {len(pdf_files)} документов (параллельно, {os.cpu_count()} ядер)...")
    t_start = time.time()

    documents, metadatas, ids = [], [], []

    with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        for pdf_path, chunks, dt in executor.map(parse_one, pdf_files):
            fname = os.path.basename(pdf_path)
            flag = "🟢" if dt < 5 else ("🟡" if dt < 20 else "🔴")
            print(f" {flag} {fname}: {dt:.1f} сек, {len(chunks)} чанков")

            for chunk in chunks:
                documents.append(chunk.text)
                ids.append(chunk.chunk_id)
                metadatas.append({
                    "source_id": chunk.metadata.source_id,
                    "title": chunk.metadata.title,
                    "source_type": chunk.metadata.source_type
                })

    print(f"\nПарсинг завершён за {time.time() - t_start:.1f} сек.")
    print(f"Извлечено {len(documents)} фрагментов. Начинаем векторизацию...")

    t0 = time.time()
    embeddings = embed_model.encode(
        documents, batch_size=64, show_progress_bar=True
    ).tolist()
    print(f"Векторизация заняла {time.time() - t0:.1f} сек")

    collection.add(
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )

    print("\nБаза знаний успешно заполнена реальными статьями и готова к поиску!")

Загрузка модели эмбеддингов (GPU)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Инициализация ChromaDB...

Начинаем парсинг 5 документов (параллельно, 48 ядер)...
 🟡 geokniga-tehnologiyaobogashcheniyapoleznyhiskopaemyh.pdf: 12.3 сек, 36 чанков
 🔴 geokniga-flotacionnye-metody-obogashcheniya_0.pdf: 103.1 сек, 304 чанков
 🟢 tehnologiya_izvlecheniya_zolota_i_serebra_iz_upornogo_zolotosoderzhaschego.pdf: 4.5 сек, 8 чанков
 🔴 geokniga_lodeyshchikovvvtehnologiyaizvlecheniyazolotaiserebraizupornyh1.pdf: 84.5 сек, 0 чанков
 🔴 geokniga-metallurgiya-blagorodnyh-metallov_0.pdf: 69.3 сек, 261 чанков


KeyboardInterrupt: 

In [1]:
# GPU версия парсера
import os
import sys
import glob
import time
import torch
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from hypothesis_factory.ingestion_gpu import GpuDoclingReader

load_dotenv()
API_KEY = os.getenv("YANDEX_API_KEY")
FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
if not API_KEY or not FOLDER_ID:
    raise ValueError("Ключи не найдены! Проверь файл .env")

# ==========================================================
# 0. ДИАГНОСТИКА: проверяем, что PyTorch реально видит GPU
# ==========================================================
print("=" * 60)
print("ПРОВЕРКА GPU")
print("=" * 60)
cuda_ok = torch.cuda.is_available()
print(f"torch.cuda.is_available(): {cuda_ok}")
if cuda_ok:
    print(f"Устройство: {torch.cuda.get_device_name(0)}")
    print(f"Свободно VRAM: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB "
          f"из {torch.cuda.mem_get_info()[1] / 1024**3:.1f} GB")
else:
    print("⚠️  CUDA НЕДОСТУПНА. Проверьте установку torch с поддержкой CUDA:")
    print("    pip install torch --index-url https://download.pytorch.org/whl/cu121")
    print("    (версию cu121 подберите под вашу версию CUDA driver)")

DEVICE = "cuda" if cuda_ok else "cpu"

# ==========================================================
# 1. Модель эмбеддингов
# ==========================================================
print("\nЗагрузка модели эмбеддингов...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device=DEVICE)
print(f"Эмбеддинг-модель загружена на: {embed_model.device}")

# ==========================================================
# 2. ChromaDB
# ==========================================================
print("\nИнициализация ChromaDB...")
client = chromadb.Client()
try:
    client.delete_collection("materials_db_notebook")
except Exception:
    pass
collection = client.create_collection("materials_db_notebook")

# ==========================================================
# 3. Docling reader с ЯВНЫМ указанием GPU
# ==========================================================
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.datamodel.base_models import InputFormat
from docling.chunking import HierarchicalChunker

print("\nНастройка Docling с GPU-ускорителем...")

accelerator_options = AcceleratorOptions(
    num_threads=6,  # CPU слабый — не даём Docling плодить лишние потоки
    device=AcceleratorDevice.CUDA if cuda_ok else AcceleratorDevice.CPU,
)

pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options = accelerator_options
pipeline_options.do_ocr = False              # оставляем — в PDF могут быть сканы
pipeline_options.do_table_structure = False  # если таблицы не нужны, можно поставить False

_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
_chunker = HierarchicalChunker()

# Подменяем внутренние объекты у уже готового ридера,
# чтобы не трогать файл ingestion_gpu.py
reader = GpuDoclingReader()
reader.converter = _converter
reader.chunker = _chunker

print(f"Docling будет использовать устройство: {accelerator_options.device}")

# ==========================================================
# 4. Парсинг с замером времени по каждому файлу
# ==========================================================
raw_data_dir = os.path.join(project_root, "data", "raw")
pdf_files = glob.glob(os.path.join(raw_data_dir, "*.pdf"))
if not pdf_files:
    raise ValueError(f"PDF файлы не найдены в папке {raw_data_dir}!")

print(f"\nНачинаем парсинг {len(pdf_files)} документов...")
documents = []
metadatas = []
ids = []

for pdf_path in pdf_files:
    fname = os.path.basename(pdf_path)
    t0 = time.time()
    chunks = reader.read(pdf_path)
    dt = time.time() - t0

    speed_flag = "🟢" if dt < 15 else ("🟡" if dt < 60 else "🔴")
    print(f" {speed_flag} {fname}: {dt:.1f} сек, {len(chunks)} чанков")

    for chunk in chunks:
        documents.append(chunk.text)
        ids.append(chunk.chunk_id)
        metadatas.append({
            "source_id": chunk.metadata.source_id,
            "title": chunk.metadata.title,
            "source_type": chunk.metadata.source_type
        })

print(f"\nИзвлечено {len(documents)} фрагментов. Начинаем векторизацию...")

# ==========================================================
# 5. Эмбеддинги
# ==========================================================
t0 = time.time()
embeddings = embed_model.encode(
    documents,
    batch_size=64 if cuda_ok else 16,  # на GPU можно батч побольше
    show_progress_bar=True,
).tolist()
print(f"Векторизация заняла {time.time() - t0:.1f} сек")

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("\nБаза знаний успешно заполнена реальными статьями и готова к поиску!")

ПРОВЕРКА GPU
torch.cuda.is_available(): True
Устройство: NVIDIA A100 80GB PCIe MIG 2g.20gb
Свободно VRAM: 19.3 GB из 19.5 GB

Загрузка модели эмбеддингов...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Эмбеддинг-модель загружена на: cuda:0

Инициализация ChromaDB...

Настройка Docling с GPU-ускорителем...
Docling будет использовать устройство: AcceleratorDevice.CUDA

Начинаем парсинг 5 документов...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

 🟢 geokniga-tehnologiyaobogashcheniyapoleznyhiskopaemyh.pdf: 6.7 сек, 238 чанков
 🟡 geokniga-flotacionnye-metody-obogashcheniya_0.pdf: 45.8 сек, 2386 чанков
 🟢 tehnologiya_izvlecheniya_zolota_i_serebra_iz_upornogo_zolotosoderzhaschego.pdf: 1.2 сек, 91 чанков
 🟡 geokniga_lodeyshchikovvvtehnologiyaizvlecheniyazolotaiserebraizupornyh1.pdf: 32.2 сек, 0 чанков
 🟡 geokniga-metallurgiya-blagorodnyh-metallov_0.pdf: 32.3 сек, 2358 чанков

Извлечено 5073 фрагментов. Начинаем векторизацию...


Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Векторизация заняла 67.0 сек

База знаний успешно заполнена реальными статьями и готова к поиску!


In [3]:
import requests

def generate_hypothesis_pipeline(problem, constraints):
    print(f"ЗАПУСК ПАЙПЛАЙНА")
    print(f"Проблема: {problem}")
    print(f"Ограничения: {constraints}\n")
    
    print("Ищем релевантный контекст в базе...")

    query_text = f"{problem}. Ограничения: {constraints}"
    query_vector = embed_model.encode(query_text).tolist()
    
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=2
    )
    
    retrieved_context = "\n- ".join(results['documents'][0])
    print("Найденный контекст:")
    print(f"- {retrieved_context}\n")
    
    print("Отправка промпта в Yandex AI Studio...")
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    system_prompt = (
        "Ты — научный ассистент НИИ. Твоя задача — сгенерировать проверяемую научную гипотезу "
        "строго на основе предоставленного контекста. "
        "Ответ структурируй: 1. Гипотеза 2. Обоснование (с опорой на контекст) 3. Риски 4. Ожидаемый эффект."
    )
    
    user_prompt = f"Проблема: {problem}\nОграничения: {constraints}\n\nКонтекст из базы знаний:\n{retrieved_context}"
    
    payload = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {"stream": False, "temperature": 0.2, "maxTokens": 2000},
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt}
        ]
    }
    
    headers = {"Content-Type": "application/json", "Authorization": f"Api-Key {API_KEY}"}
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 200:
        answer = response.json()['result']['alternatives'][0]['message']['text']
        print("Ответ модели:\n")
        print("="*50)
        print(answer)
        print("="*50)
    else:
        print(f"Ошибка API {response.status_code}: {response.text}")

target_problem = "Уменьшения вредного влияния на процесс цианирования руды сурьмянистых и мышьяковистых минералов"
user_constraints = "Бюджет ограничен, необходимо избегать дорогих компонентов"

# Запуск
generate_hypothesis_pipeline(target_problem, user_constraints)

ЗАПУСК ПАЙПЛАЙНА
Проблема: Уменьшения вредного влияния на процесс цианирования руды сурьмянистых и мышьяковистых минералов
Ограничения: Бюджет ограничен, необходимо избегать дорогих компонентов

Ищем релевантный контекст в базе...
Найденный контекст:
- Для уменьшения вредного влияния на процесс цианирования ру -ды сурьмянистых и мышьяковистых минералов следует :
- - -снизить до минимума концентрацию защитной щелочи
- -максимально  быстро превращать  вредные тиосоли и сульфид -
- ионы в относительно безвредные роданид -ионы введением в процесс растворимых  солей  свинца -азотно -кислого  или  уксусно -кислого . Вместо указанных  солей свинца  можно  применять  более дешевый

Отправка промпта в Yandex AI Studio...
Ошибка API 429: {"error":{"grpcCode":8,"httpCode":429,"message":"ai.textGenerationCompletionSessionsCount.count gauge quota limit exceed: allowed 10 requests","httpStatus":"Too Many Requests","details":[]}}
